In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
import os
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL", "")

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [10]:
import re
import ast
import json
from statistics import mean

def validate_json(text):
    """ Parse output as JSON """
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    """ Parse output to a Python Abstract Syntax Tree """
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    """ Compile output as a Regex """
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    The response must be valid JSON. Escape every backslash as \\\\ and do not
    quote code or regex patterns in your text fields.
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    final_score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "reasoning": reasoning,
        "syntax_score": syntax_score,   
        "final_score": final_score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Calculate an average score across all test cases
    average_score = mean([result["final_score"] for result in results])
    print(f"Average score: {average_score}") 
    
    return results

In [11]:
with open("011_dataset_2.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# The evaluation returns a structured JSON array where each object represents one test case result
print(json.dumps(results, indent=2))

Average score: 5.833333333333333
[
  {
    "output": "\nimport re\n\ndef extract_account_id(arn):\n    match = re.search(r'^arn:aws:[^:]+:([^:]*):([^:]*)', arn)\n    if match:\n        account_id = match.group(2)\n        return account_id if account_id else None\n    return None\n",
    "test_case": {
      "task": "Extract the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/path/to/object'",
      "format": "Regex"
    },
    "model_score": 3,
    "reasoning": "The solution has a fundamental flaw: S3 ARNs like arn:aws:s3:::my-bucket/path/to/object have empty region and account fields by design. The regex will match correctly, but both capture groups will be empty strings, causing the function to return None even for valid S3 ARNs. For other services like IAM or EC2 that do include account IDs, the function would work, but there's no service detection or documentation of this limitation. A proper solution should either (1) document that it only works for specific servic